In [ ]:
import os
import json

import networkx as nx
from word_nodes import WordNode, MetaNode
import nltk
from nltk.corpus import wordnet as wn
from tqdm import tqdm

from helpers import nltk_to_wordnet, tokenize_1, replace_contractions, wordnet_pos, get_correct_pos_tag

In [ ]:
venv_nltk_path = os.path.join(os.getcwd(), '../.venv', 'nltk_data')

if not os.path.exists(venv_nltk_path):
    os.makedirs(venv_nltk_path)

nltk.download('averaged_perceptron_tagger_eng', download_dir=venv_nltk_path)
nltk.download('wordnet', download_dir=venv_nltk_path)
nltk.download('omw-1.4', download_dir=venv_nltk_path)
nltk.download('punkt_tab', download_dir=venv_nltk_path)

### Helper Functions

In [ ]:
def len_of_split(string):
    a = string.split('_')
    return len(a)

In [ ]:
def preprocess_text(text):
     return replace_contractions(text)

In [ ]:
def get_wordnet_usable_tokens(pos_tagged_tokens):
    wordnet_usable_tokens = set()
    for token, tag in pos_tagged_tokens:
        if tag in nltk_to_wordnet.keys():
            wordnet_tag = nltk_to_wordnet[tag]
            # if pos_tag matches a definition
            if len(wn.synsets(token, pos=wordnet_tag[0])) > 0 or (wordnet_tag[0] == 'a' and len(wn.synsets(token, pos='s')) > 0):
                wordnet_usable_tokens.add((token.lower(), wordnet_tag))

            # otherwise give an unknown pos_tag
            else:
                all_synsets = wn.synsets(token)
                if len(all_synsets) > 0:
                    first_pos = all_synsets[0].pos()
                    if all([syn.pos() == first_pos for syn in all_synsets]):       # if there is only one pos, assume that that is the correct one
                        wordnet_usable_tokens.add((token.lower(), wordnet_pos[first_pos]))
                    else:
                        wordnet_usable_tokens.add((token.lower(), ("UNK_POS",)))

    return wordnet_usable_tokens
        

In [ ]:
def verify_pos_tag(word, nltk_assigned_pos_tag):
    if nltk_assigned_pos_tag not in nltk_to_wordnet:
        return False
    
    valid_pos_tags = set([synset.pos() for synset in wn.synsets(word)])
    pos = nltk_to_wordnet[nltk_assigned_pos_tag]

    if pos not in valid_pos_tags:
        return False
        
    return (word, pos)

### Get List of Tokens

In [ ]:
example_sentences = []
for synset in wn.all_synsets():
    for sentence in synset.examples():
        example_sentences.append(sentence)

In [ ]:
def get_tokens_from_wordnet(tokens=None):
    _tokens = set() if not tokens else tokens

    # tokenize and process wordnet definitions
    all_synsets_loop = tqdm(wn.all_synsets(), f'processing wordnet definitions...')
    for synset in all_synsets_loop:
        all_synsets_loop.set_description(f'processing wordnet definitions... Tokens Found: {len(_tokens)}')
        # tag individual lemmas
        lemmas = synset.lemmas()
        for lemma in lemmas:
            lemma_pos_tag = get_correct_pos_tag(lemma.name(), synset.pos())
            
            _tokens.update([(lemma.name().lower(), lemma_pos_tag)])

        # tag example sentences to get tokens
        for sentence in synset.examples():
            sent_tokenized = tokenize_1(sentence)
            for lemma in [l.name() for l in lemmas if l.name() in sent_tokenized]:
                for l2 in lemmas:
                    new_sent = [(word if word != lemma else l2.name()) for word in sent_tokenized]
                    example_pos_tagged = nltk.pos_tag(new_sent)
                    example_tokens = get_wordnet_usable_tokens(example_pos_tagged)
                    _tokens.update(example_tokens)

        # tag definitions to get tokens
        definition = tokenize_1(preprocess_text(synset.definition()))
        _tokens.update(get_wordnet_usable_tokens(nltk.pos_tag(definition)))

    return _tokens

In [ ]:
def get_tokens_from_corpus(tokens=None, files=None):
    _tokens = set() if not tokens else tokens
    
    file_list = [txt_file for txt_file in os.listdir('Corpus') if '.txt' in txt_file] if files is None else files
    loop = tqdm(total=len(file_list))
    for txt_file in file_list:
        # open file and replace all contractions with expanded form
        loop.set_description(f'processing {txt_file} - opening file... Tokens Found: {len(_tokens)}')
        with open(os.path.join('Corpus', txt_file), 'r', encoding='utf-8') as file:
            text = preprocess_text(file.read())
    
        # get tokens that have a synset in wordnet (to get a better list of all forms of different words)
        loop.set_description(f'processing {txt_file} - tokenizing... Tokens Found: {len(_tokens)}')
        corpus_tokens = tokenize_1(text)                 # tokenize corpus

        loop.set_description(f'processing {txt_file} - pos_tagging... Tokens Found: {len(_tokens)}')
        corpus_tagged_tokens = nltk.pos_tag(corpus_tokens)    # tokenize_corpus
        
        corpus_tokens = get_wordnet_usable_tokens(corpus_tagged_tokens)
        
        _tokens.update(corpus_tokens)
        loop.update()

    loop.set_description(f'Total Tokens: {len(_tokens)}; complete.')
    loop.close()

    return _tokens
    

In [ ]:
def get_network_tokens(corpus_files=None):
# tokenize and process wordnet definitions
    print('Getting tokens from WordNet:')
    tokens1 = get_tokens_from_wordnet()

    # add tokens from corpus
    print('\nGetting tokens from corpus:')
    tokens2 = get_tokens_from_corpus(files=corpus_files)
                           
    print('\nUnioning Tokens...')
    tokens = tokens1.union(tokens2)

    print(f'Total tokens: {len(tokens)}')
    return tokens


In [ ]:
tokens = get_network_tokens()

tokens_list = list(tokens)
tokens_set = set(tokens_list)
tokens_list[0:20]

In [ ]:
file_path = 'data/tokens_raw.json'
with open(file_path, 'w') as file:
    json.dump(tokens_list, file)

### Make Graph

#### Get Tokens

In [ ]:
tokens_file_path = 'data/tokens_raw.json'

In [ ]:
def read_in_raw_tokens(file_path=tokens_file_path):
    with open(file_path, 'r') as file:
        tokens_json = json.load(file)
        
    for i in range(len(tokens_json)):
        token = tokens_json[i]
        token[1] = tuple(token[1])
        tokens_json[i] = tuple(tokens_json[i])
        
    tokens_set = set(tokens_json)
    return tokens_set
    

In [ ]:
tokens_set = read_in_raw_tokens()

#### Create Nodes

In [ ]:
# Create nodes from tokens

nodes = {}
for token in tqdm(tokens_set, 'creating nodes...'):
    word, meta = token
    pos = meta[0]
    
    node = WordNode(token)

    if pos != "UNK_POS":
        for syn in wn.synsets(word, pos=pos):
            node.add_definition(syn)
    nodes[token] = node

#### Create Meta Nodes

In [ ]:
meta_nodes = {'singular': MetaNode('singular', opposites=['plural']),
              'plural': MetaNode('plural', opposites=['singular']),
              'common': MetaNode('common', opposites=['proper']),
              'proper': MetaNode('proper', opposites=['common']),
              'base': MetaNode('base', opposites=['participle']),
              'past': MetaNode('past', opposites=['present']),
              'present': MetaNode('present', opposites=['past']),
              '3rd-person': MetaNode('3rd-person', opposites=['non-3rd-person']),
              'non-3rd-person': MetaNode('non-3rd-person', opposites=['3rd-person']),
              'participle': MetaNode('participle', opposites=['base']),
              'comparative': MetaNode('comparative', opposites=['superlative']),
              'superlative': MetaNode('superlative', opposites=['comparative'])}

In [ ]:
# Add Meta Nodes to graph
graph = nx.DiGraph()
for key, meta_node in meta_nodes.items():
    graph.add_node(meta_node, label=(key, 'meta'), value=key, type='meta')
    graph.add_edge(meta_node, meta_node, weight=1)      # add a self loop to prevent premature path endings

#### Add Word Nodes and Edges to Graph

In [ ]:
for key, node in tqdm(nodes.items(), 'creating graph...'):
    word, pos = key
    pos, *meta = pos
    
    graph.add_node(node, label=key, value=word, type='word')
    for meta_node_label in meta:
        graph.add_edge(node, meta_nodes[meta_node_label], weight=1)
        
    for synset, definition in node.definitions.items():
        definition_tokens = get_wordnet_usable_tokens(nltk.pos_tag(tokenize_1(definition)))
        for tok in definition_tokens:
            other = nodes[tok]
            if not graph.has_edge(node, other):
                graph.add_edge(node, other, weight=1)
            else:
                graph[node][other]['weight'] += 1
                

In [ ]:
# remove self-loops (Except for on Meta Nodes)
node_types = nx.get_node_attributes(graph, 'type')

count = 0
for node in tqdm(graph.nodes(), 'removing self-loops...'):
    if graph.has_edge(node, node) and node_types[node] != 'meta':
        graph.remove_edge(node, node)
        count += 1
        
print(f'{count} self-loops found and removed')

In [ ]:
nx.write_gexf(graph, 'data/pos_tagged_graph_original.gexf')

#### Update Edge Weights (Create Weighted Graph)

In [ ]:
graph_copy = nx.read_gexf('data/pos_tagged_graph_original.gexf')

In [ ]:
# weight each edge proportional to the in-degree of the node it leads to 
node_types = nx.get_node_attributes(graph_copy, 'type')
for node in tqdm(graph_copy.nodes(), 'updating_weights...'):
    in_edge_attributes = [edge[2] for edge in graph_copy.in_edges(node, data=True)]
    in_degree = sum([item['weight'] for item in in_edge_attributes])
    for edge_attributes in in_edge_attributes:
        edge_attributes['weight'] = edge_attributes['weight'] / in_degree if node_types[node] != 'meta' else 0.05

In [ ]:
# convert weights on edges to probabilities
for node in tqdm(graph_copy.nodes(), 'calculating probabilities...'):
    out_edge_attributes = [edge[2] for edge in graph_copy.out_edges(node, data=True)]
    total_out = sum([(item['weight']) for item in out_edge_attributes])
    for edge_attribute in out_edge_attributes:
        edge_attribute['weight'] = (edge_attribute['weight']) / total_out

In [ ]:
nx.write_gexf(graph_copy, 'data/graph_weighted_with_probs.gexf')